# jupyter_ai_persona_manager smoke test

Exercises the public API surface re-exported from `jupyter_ai_persona_manager.__init__`:
Pydantic MCP settings models, `PersonaDefaults`, `BasePersona` ABC, and the extension/manager symbols.

In [ ]:
# Example 1: the public API re-exported from jupyter_ai_persona_manager.__init__
import jupyter_ai_persona_manager

from jupyter_ai_persona_manager import (
    BasePersona,
    PersonaDefaults,
    PersonaManager,
    PersonaRequirementsUnmet,
    ContractLevel,
    McpSettings,
    McpServerHttp,
    McpServerStdio,
    PersonaManagerExtension,
)

assert not hasattr(jupyter_ai_persona_manager, "PersonaAwareness"), (
    "PersonaAwareness is re-exported again -- it was removed at 0.2.0; if upstream brought it "
    "back, put it back in the list below instead of leaving this assertion to fail"
)

exported = [
    BasePersona, PersonaDefaults, PersonaManager, PersonaRequirementsUnmet,
    ContractLevel, McpSettings, McpServerHttp, McpServerStdio, PersonaManagerExtension,
]
for sym in exported:
    print(f"{sym.__module__}.{sym.__name__}")

assert all(isinstance(s, type) for s in exported), "all re-exports should be classes"
assert len(exported) == 9, len(exported)

In [ ]:
# Example 2: PersonaDefaults is a Pydantic model with required + optional fields.
from jupyter_ai_persona_manager import PersonaDefaults

defaults = PersonaDefaults(
    name="TestBot",
    description="A test persona for smoke checks",
    avatar_path="/tmp/avatar.svg",
    system_prompt="You are a helpful test assistant.",
)

print(f"name={defaults.name!r}")
print(f"model_uid (optional) = {defaults.model_uid!r}")
print(f"slash_commands type = {type(defaults.slash_commands).__name__}")

dumped = defaults.model_dump()
assert dumped["name"] == "TestBot"
assert dumped["system_prompt"].startswith("You are")
assert dumped["model_uid"] is None

In [ ]:
# Example 3: MCP stdio + HTTP server descriptor models round-trip through JSON.
import json

from jupyter_ai_persona_manager import McpServerStdio, McpServerHttp

stdio = McpServerStdio(
    name="local-tools",
    command="/usr/bin/python3",
    args=["-m", "my_mcp_server"],
    env=[{"name": "DEBUG", "value": "1"}],
)
http = McpServerHttp(
    type="http",
    name="remote-tools",
    url="https://example.invalid/mcp",
    headers=[{"name": "X-API-Key", "value": "secret"}],
)

blob = json.loads(stdio.model_dump_json())
assert blob["command"] == "/usr/bin/python3"
assert blob["env"][0]["name"] == "DEBUG"
assert http.type == "http"
assert http.url.startswith("https://")
print("stdio:", stdio.model_dump_json())
print("http :", http.model_dump_json())

In [ ]:
# Example 4: McpSettings holds a discriminated list of stdio + http servers.
from jupyter_ai_persona_manager import McpServerHttp, McpServerStdio, McpSettings

settings = McpSettings(mcp_servers=[
    McpServerStdio(name="a", command="/bin/true", args=[]),
    McpServerHttp(type="http", name="b", url="https://example.invalid"),
])

assert len(settings.mcp_servers) == 2
assert isinstance(settings.mcp_servers[0], McpServerStdio)
assert isinstance(settings.mcp_servers[1], McpServerHttp)

# Round-trip through dict -> model.
raw = settings.model_dump()
rebuilt = McpSettings(**raw)
assert len(rebuilt.mcp_servers) == 2
print("OK round-trip: 2 servers")

In [ ]:
# Example 5: BasePersona is an abstract class + the extension exposes Jupyter hooks.
import inspect

import jupyter_ai_persona_manager
from jupyter_ai_persona_manager import BasePersona, PersonaManagerExtension, PersonaRequirementsUnmet

# BasePersona is abstract; instantiation must fail.
assert inspect.isabstract(BasePersona), "BasePersona should be abstract"
try:
    BasePersona()
except TypeError as e:
    print(f"expected TypeError on abstract instantiation: {e}")

# PersonaRequirementsUnmet is an exception class.
assert issubclass(PersonaRequirementsUnmet, Exception)

# Jupyter server / lab extension entry points are defined at the package level.
server_eps = jupyter_ai_persona_manager._jupyter_server_extension_points()
lab_eps = jupyter_ai_persona_manager._jupyter_labextension_paths()
assert server_eps and server_eps[0]["app"] is PersonaManagerExtension
assert lab_eps and lab_eps[0]["dest"] == "@jupyter-ai/persona-manager"
print("server ext:", server_eps)
print("lab ext   :", lab_eps)